In [ ]:
%pip install --upgrade nbformat plotly -qqq

import torch
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
import sys
sys.path.append('../CaloChallenge/code/')
from HighLevelFeatures import HighLevelFeatures as HLF
import os

1068.97s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


1074.92s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [54]:
filename = "../calo-data/FCCeeALLEGRO/testing/LEMURS_FCCeeALLEGRO_gamma_1000events_5GeV_phi0.0_theta1.57.h5"
# filename = "/home/panos/turing/diffusion4sim/simulations/generated_1000events_Geo_SiW_E_50GeV_Phi_0.0_Theta_2.1.h5"
# filename = "/home/panos/turing/fastsim/generated_1000events_Geo_SiW_E_50GeV_Phi_0.0_Theta_1.57.h5"

# i_idcs = None
i_idcs = np.arange(0,100)

with h5py.File(filename, "r") as f:
    print("Features in " + os.path.basename(filename) + ":", list(f.keys()))
    
    showers = f["showers"][i_idcs,:] if i_idcs is not None else f["showers"]
    showers = np.array(showers)
    # showers = showers.reshape(showers.shape[0],-1)
    incident_energies = f["incident_energy"][i_idcs] if i_idcs is not None else f["incident_energy"]
    incient_energies = np.array(incident_energies).reshape(-1,1)

print("Showers shape:", showers.shape)
print("Incident energies shape:", incident_energies.shape)

# i_idx = 0
# shower3d = showers[i_idx, :, :, :]
# import pandas
# df = pandas.DataFrame(shower3d.flatten())
# df.describe()

Features in LEMURS_FCCeeALLEGRO_gamma_1000events_5GeV_phi0.0_theta1.57.h5: ['incident_energy', 'incident_phi', 'incident_theta', 'showers']
Showers shape: (100, 9, 16, 45)
Incident energies shape: (100,)


In [48]:
# cell_hits: [N, num_ang, num_rad, num_layers]
def get_guiding_field(cell_hits):
    # Mean energy per cell across all events
    # Shape: [num_ang, num_rad, num_layers]
    mean_profile = torch.mean(cell_hits, dim=0)
    
    # Normalize each layer so it represents the expected 2D shape at that depth
    # Sum over ang and rad dimensions (dims 0 and 1)
    layer_sums = mean_profile.sum(dim=(0, 1), keepdim=True)
    return mean_profile / (layer_sums + 1e-9)

mean_rho = get_guiding_field(torch.tensor(showers))

# Shower SDE

Let $E(x,t)$ be the energy density at position x at time t. Let $ D(x,t) $ be the deposited at cell in position x at time t for which

$$ 
\dfrac{\partial D(x,t)}{\partial t} = \alpha E(x,t)
$$

Let

$$
V(x,t) = -ln(\bar\rho(x,E_0))
$$

be the drift potential for the mean density profile. 

Let following SDE to model a shower's development

$$
dE = \big[ -v \cdot \nabla E -\lambda( E - \bar \rho) \big] dt + \sigma(E,x)dW
$$

$-v \cdot \nabla E$: advection\
$-\lambda( E - \bar \rho)$: mean-reverting froce\
$ \sigma(E,x)dW$: noise term

## Conservation of energy constraint

If it holds $ E_0 = \int E(x,t)~dx + \int D(x,t)~dx $, then we can enforce that

$$ \int \sigma(E,x)~dWdx = 0 $$


In [ ]:
import torch
import torch.nn as nn
from torch.nn.functional import interpolate as interp

class CaloSDE(nn.Module):
    def __init__(self, mean_rho, lmbda=0.5, sigma=0.1, alpha=0.2):
        super().__init__()
        # mean_rho: [num_layers, num_ang, num_rad]
        # We register it as a buffer so it moves with the model to GPU
        self.register_buffer("mean_rho", mean_rho)
        
        # Learnable or fixed parameters
        self.lmbda = nn.Parameter(torch.tensor(lmbda))  # Mean reversion strength
        self.sigma = nn.Parameter(torch.tensor(sigma))  # Noise scale
        self.alpha = nn.Parameter(torch.tensor(alpha))  # Deposition rate
        
        self.n_layers, self.n_ang, self.n_rad = mean_rho.shape
        self.noise_type = "diagonal"
        self.sde_type = "ito"


    def get_mean_at_t(self, t):
        """Helper to interpolate the mean profile at continuous time t."""
        # Convert t (0 to T) to layer index (0 to n_layers-1)
        idx = torch.clamp(t, 0, self.n_layers - 1).long()
        # Returns [batch, num_ang, num_rad]
        return self.mean_rho[idx]

    def h(self, t, y):

        # Reshape y from flat to [batch, ang, rad]
        y_grid = y.view(-1, self.n_ang, self.n_rad)
        
        # 1. Get the target shape for the current layer
        target_shape = self.get_mean_at_t(t)
        
        # 2. Calculate the 'current' energy sum to scale the target shape
        current_energy_sum = y_grid.sum(dim=(1, 2), keepdim=True)
        expected_y = current_energy_sum * target_shape
        
        # 3. Combine Drift (Mean Reversion) and Deposition (Energy Loss)
        drift = -self.lmbda * (y_grid - expected_y) - self.alpha * y_grid
        
        return drift.view(y.shape)

    def g(self, t, y):
        """
        Diffusion Term: 
        Models shot noise/Poisson fluctuations inherent in particle showers.
        """
        # We use sqrt(abs(y)) to ensure the noise remains real-valued
        diffusion = self.sigma * torch.sqrt(torch.abs(y) + 1e-9)
        return diffusion
    

import torchsde

n_layers = 45

# Initial state y0: [Batch, Ang * Rad]
# This should be your incident energy distributed per the first layer of mean_rho
y0 = torch.tensor(incident_energies[0]) * mean_rho[0].flatten()

# Time points (one per layer)
t_steps = torch.linspace(0, n_layers - 1, n_layers)

# Solve the SDE
# result shape: [n_layers, Batch, Ang * Rad]
result = torchsde.sdeint(CaloSDE(mean_rho), y0, t_steps)

# To find energy deposited AT each layer:
# Energy_dep[z] = alpha * result[z]
energy_deposited = params['alpha'] * result

ValueError: `y0` must be a 2-dimensional tensor of shape (batch, channels).